[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdgordillob/aca_aci_collab/blob/main/notebooks/aca_opt_multi_region_monthly.ipynb)

# aca_opt multi-region monthly anomalies (temperature + wind + precipitation + drought)

Runs `calcular_anomalias_regiones.py`'s existing multi-component orchestrator -- `procesar_anomalias_region()`, already handling temperature (T90/T10), wind power (WP), precipitation (Rx5day), and drought (CDD) together per region -- against the full raw ERA5 archive fetched from Drive, for the regions this pipeline documents as actually populated (national, Antioquia, Cundinamarca-Bogota, Valle del Cauca). No new pipeline logic: this wires existing, already-importable functions to Drive-sourced data across three variables instead of one.

**This is a long run.** Fetching three variables (temperature, precipitation, wind) x 64 years is roughly 3x `aca_opt_full_replication.ipynb`'s ~5.5GB, and Stage 3 runs three anomaly calculations per region. Expect **2.5-3.5 hours total** for the default 4 regions -- test with `YEARS = range(1961, 1965)` and `REGIONS = REGIONS[:1]` first if you just want to confirm it runs before committing to the full range.

**Known pre-existing bug this notebook works around, not fixes:** `calcular_percentil_lluvia.py` saves its output as `era5_lluvia_percentil.nc` (singular), but `calcular_anomalias_lluvia.py` reads `era5_lluvias_percentil.nc` (plural) -- a naming mismatch already visible in this repo's own `data/processed/` (both filenames exist there, side by side). This notebook copies the output under both names rather than silently hiding the inconsistency.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_COLAB

## 1. Get the code

In [ ]:
import sys, os

REPO_ROOT = "/content/aca_indice_climatico_opt"

if IN_COLAB:
    !git clone --depth 1 https://github.com/mdgordillob/aca_indice_climatico_opt.git {REPO_ROOT}
else:
    REPO_ROOT = os.path.abspath("../../aca_indice_climatico_opt-main")  # adjust if running locally

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "scripts"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "utils"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))  # for drive_sync
os.chdir(REPO_ROOT)  # calcular_percentil_lluvia.calcular_percentiles() takes no args -- resolves paths from __file__, needs cwd/repo consistent

## 2. Install dependencies (pinned to requirements.txt)

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y libeccodes-dev > /dev/null
    !pip install -q cfgrib==0.9.14.1 eccodes==2.39.1 rioxarray==0.18.2 geopandas==1.0.1 netCDF4==1.7.2 xarray==2024.11.0

## 3. Mount Drive and fetch everything

All shapefiles (small, ~a few MB total) plus `era5_{tmp,rain,wind}_<year>.grib` for every year in `YEARS`. Uses conventional repo-relative paths throughout (`data/raw/era5/`, `data/processed/...`) rather than a scratch directory -- this is a fresh clone, so there's nothing to collide with, and several of the scripts below (`calcular_percentil_lluvia.calcular_percentiles()`) hardcode those conventional paths rather than accepting overrides.

In [ ]:
import shutil, time

DRIVE_ROOT = "/content/drive/MyDrive/2. Datos"
YEARS = range(1961, 2025)  # narrow this for a quick test run, e.g. range(1961, 1965)

raw_dir = os.path.join(REPO_ROOT, "data", "raw", "era5")
os.makedirs(raw_dir, exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    import drive_sync
    drive_sync.sync(DRIVE_ROOT, REPO_ROOT, only=["shapefiles"])

    completos = os.path.join(DRIVE_ROOT, "era5", "completos")
    t0 = time.perf_counter()
    fetched = {"tmp": 0, "rain": 0, "wind": 0}
    for var in fetched:
        for year in YEARS:
            name = f"era5_{var}_{year}.grib"
            src = os.path.join(completos, name)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(raw_dir, name))
                fetched[var] += 1
    print(f"fetched {fetched} in {time.perf_counter()-t0:.0f}s")
else:
    print("Not in Colab -- assuming data/raw/era5 and data/shapefiles are already populated locally.")

## 4. Stage 1 -- merge/resample the 1961-1990 baseline, all three variables

Only the baseline years, for the same reason as `aca_opt_full_replication.ipynb`: Stage 2 for every variable only ever reads the `1961-1990` slice.

In [ ]:
import unir_archivos

processed_dir = os.path.join(REPO_ROOT, "data", "processed")
baseline_years = [y for y in range(1961, 1991) if y in YEARS]

def run_stage1(var_key, process_one_year, output_subdir, merged_name, merge_variable):
    # process_one_year: (grib_path, year, output_dir) -> None -- a thin adapter per
    # variable below, since process_yearly_data_tmp/process_yearly_precipitation_data
    # take (file_path, year, variable, output_dir) but process_yearly_wind_data takes
    # (file_path, year, output_dir, variable) -- inconsistent argument order between
    # the three, not something to paper over with one "uniform" call.
    out_dir = os.path.join(processed_dir, output_subdir)
    present = [y for y in baseline_years if os.path.exists(os.path.join(raw_dir, f"era5_{var_key}_{y}.grib"))]
    print(f"{var_key}: {len(present)}/{len(baseline_years)} baseline years present")
    t0 = time.perf_counter()
    for year in present:
        process_one_year(os.path.join(raw_dir, f"era5_{var_key}_{year}.grib"), year, out_dir)
    unir_archivos.merge_yearly_files(out_dir, os.path.join(processed_dir, merged_name), merge_variable)
    print(f"Stage 1 ({var_key}, {len(present)} years): {(time.perf_counter()-t0)/60:.1f} min")

run_stage1(
    "tmp",
    lambda f, y, o: unir_archivos.process_yearly_data_tmp(f, y, "t2m", o),
    "daily_by_year_tmp", "era5_daily_combined_tmp.nc", "t2m",
)
run_stage1(
    "rain",
    lambda f, y, o: unir_archivos.process_yearly_precipitation_data(f, y, "tp", o),
    "daily_by_year_rain", "era5_daily_combined_rain.nc", "tp",
)
run_stage1(
    "wind",
    lambda f, y, o: unir_archivos.process_yearly_wind_data(f, y, o, ["u10", "v10"]),
    "daily_by_year_wind", "era5_daily_combined_wind.nc", ["u10", "v10"],
)

## 5. Stage 2 -- baseline percentiles, all three variables

In [ ]:
import calcular_percentil_temperatura as percentil_tmp
import calcular_percentil_viento as percentil_wind
import calcular_percentil_lluvia as percentil_rain

t0 = time.perf_counter()
est_tmp = percentil_tmp.calcular_percentiles(os.path.join(processed_dir, "era5_daily_combined_tmp.nc"))
percentil_tmp.guardar_percentiles(est_tmp, os.path.join(processed_dir, "era5_temperatura_percentil.nc"), processed_dir, guardar_csv=False)
print(f"temperature percentiles: {(time.perf_counter()-t0)/60:.1f} min")

t0 = time.perf_counter()
est_wind = percentil_wind.calcular_percentiles_viento(os.path.join(processed_dir, "era5_daily_combined_wind.nc"))
percentil_wind.guardar_percentiles_viento(est_wind, os.path.join(processed_dir, "era5_wind_percentil.nc"), processed_dir, guardar_csv=False)
print(f"wind percentiles: {(time.perf_counter()-t0)/60:.1f} min")

t0 = time.perf_counter()
percentil_rain.calcular_percentiles()  # no args -- reads/writes REPO_ROOT/data/processed/ via __file__
# work around the lluvia/lluvias naming mismatch (see intro) rather than hide it:
shutil.copy(
    os.path.join(processed_dir, "era5_lluvia_percentil.nc"),
    os.path.join(processed_dir, "era5_lluvias_percentil.nc"),
)
print(f"rain + drought percentiles: {(time.perf_counter()-t0)/60:.1f} min")

## 6. Stage 3 -- anomalies for every region, all components

`procesar_anomalias_region()` calls temperature/wind/precipitation once **per region**, each re-decoding the same raw grib from scratch -- 4 regions x 3 variables x 64 years = 768 redundant grib-decode passes for data that's identical across regions until the final shapefile clip. Temperature and wind expose a clean load-once (`load_annual_grid_data`/`load_annual_grid_data_safe`, both take an *optional* `shapefile_path`) then clip-per-region split, so the cells below decode each year **once** and clip to all 4 regions from that one decode -- roughly a 4x reduction for those two variables. Precipitation's equivalent (`calcular_anomalias_lluvia.load_grid_data`) requires a shapefile unconditionally with no split available without a more invasive change to that script, so it's left as the original per-region loop for now -- still correct, just not sped up.

**This changes performance only, not the underlying computation** -- same `calcular_anomalias`/`calcular_anomalias_viento` functions, same percentile files, same multiprocessing-over-years pattern (now with each worker handling all 4 regions for its year instead of 1). The open multiprocessing-correctness question from `ARCHITECTURE.pdf` \S9.4 is therefore still open here too -- this notebook does not resolve it, just runs faster either way.

In [ ]:
import calcular_anomalias_temperatura as anomalias_tmp
import calcular_anomalias_viento as anomalias_wind
import calcular_anomalias_lluvia as anomalias_rain
from multiprocessing import Pool, cpu_count
import xarray as xr

shapefiles_dir = os.path.join(REPO_ROOT, "data", "shapefiles")

REGIONS = [
    {"name": "anomalias_colombia", "shapefile": "colombia_4326.shp"},
    {"name": "anomalias_antioquia", "shapefile": "antioquia_4326.shp"},
    {"name": "anomalias_cundinamarca_bogota", "shapefile": "Cundinamarca_Bogota_4326.shp"},
    {"name": "anomalias_valle_cauca", "shapefile": "valle_cauca_4326.shp"},
    # Shapefiles exist for these too (uncomment to include) -- not part of
    # this repo's documented "actually populated" set, so left off by default:
    # {"name": "anomalias_bogota", "shapefile": "bogota.shp"},
    # {"name": "anomalias_medellin", "shapefile": "medellin_4326.shp"},
    # {"name": "anomalias_cali", "shapefile": "cali_4326.shp"},
    # {"name": "anomalias_san_andres_providencia", "shapefile": "san_andres_providencia.shp"},
]

for region in REGIONS:
    os.makedirs(os.path.join(processed_dir, region["name"]), exist_ok=True)

### 6a. Temperature + wind -- decode once per year, clip to all regions

In [ ]:
def _year_task_temp(args):
    year, grib_path, archivo_percentiles, regions, shapefiles_dir, processed_dir = args
    import calcular_anomalias_temperatura as m
    results = {r["name"]: [] for r in regions}
    try:
        annual = m.load_annual_grid_data(grib_path, year, "t2m", shapefile_path=None)
    except Exception as e:
        print(f"  Error loading tmp {year}: {e}")
        return results
    for region in regions:
        shp = os.path.join(shapefiles_dir, region["shapefile"])
        try:
            shape = m.get_cached_shapefile(shp)
            clipped = annual.rio.write_crs("EPSG:4326", inplace=True)
            clipped = clipped.rio.clip(shape.geometry, shape.crs, drop=True)
        except Exception as e:
            print(f"  Error clipping tmp {year} {region['name']}: {e}")
            continue
        out_dir = os.path.join(processed_dir, region["name"])
        for month in range(1, 13):
            try:
                monthly = m.get_monthly_data(clipped, year, month, "t2m")
                if len(monthly.time) == 0:
                    continue
                ds_month = m.calcular_anomalias(
                    archivo_percentiles, monthly, year, month,
                    os.path.join(out_dir, f"anomalies_temperature_{year}_{month}.nc"),
                    shapefile_path=shp,
                )
                results[region["name"]].append(ds_month.assign_coords(year=year))
            except Exception as e:
                print(f"  Error tmp {region['name']} {year}-{month}: {e}")
    return results

def _year_task_wind(args):
    year, grib_path, archivo_percentiles, regions, shapefiles_dir, processed_dir = args
    import calcular_anomalias_viento as m
    results = {r["name"]: [] for r in regions}
    annual = m.load_annual_grid_data_safe(grib_path, year, "wind_speed", shapefile_path=None)
    if annual is None:
        return results
    for region in regions:
        shp = os.path.join(shapefiles_dir, region["shapefile"])
        try:
            shape = m.get_cached_shapefile(shp)
            clipped = annual.rio.write_crs("EPSG:4326", inplace=True)
            clipped = clipped.rio.clip(shape.geometry, shape.crs, drop=True)
        except Exception as e:
            print(f"  Error clipping wind {year} {region['name']}: {e}")
            continue
        out_dir = os.path.join(processed_dir, region["name"])
        for month in range(1, 13):
            try:
                monthly = m.get_monthly_data(clipped, year, month, "wind_speed")
                if len(monthly.time) == 0:
                    continue
                ds_month = m.calcular_anomalias_viento(
                    archivo_percentiles, monthly, year, month,
                    os.path.join(out_dir, f"anomalies_wind_{year}_{month}.nc"),
                    shapefile_path=shp,
                )
                results[region["name"]].append(ds_month.assign_coords(year=year))
            except Exception as e:
                print(f"  Error wind {region['name']} {year}-{month}: {e}")
    return results

def run_variable_multi_region(var_key, task_fn, archivo_percentiles, output_filename, num_workers=None):
    files = [f for f in os.listdir(raw_dir) if f.endswith(".grib") and var_key in f]
    tasks = []
    for year in YEARS:
        matches = [f for f in files if str(year) in f]
        if len(matches) == 1:
            tasks.append((year, os.path.join(raw_dir, matches[0]), archivo_percentiles, REGIONS, shapefiles_dir, processed_dir))
    print(f"{var_key}: {len(tasks)}/{len(list(YEARS))} years present")

    num_workers = num_workers or max(1, cpu_count() - 1)
    t0 = time.perf_counter()
    with Pool(num_workers) as pool:
        year_results = pool.map(task_fn, tasks)

    combined = {r["name"]: [] for r in REGIONS}
    for yr in year_results:
        for name, items in yr.items():
            combined[name].extend(items)

    for region in REGIONS:
        items = combined[region["name"]]
        if not items:
            print(f"  no {var_key} results for {region['name']}")
            continue
        df = xr.concat(items, dim="time").to_dataframe().reset_index()
        df.to_csv(os.path.join(processed_dir, region["name"], output_filename), index=False)

    print(f"{var_key}: {(time.perf_counter()-t0)/60:.1f} min for all {len(REGIONS)} regions")

run_variable_multi_region("tmp", _year_task_temp, os.path.join(processed_dir, "era5_temperatura_percentil.nc"), "anomalies_temperature_combined.csv")
run_variable_multi_region("wind", _year_task_wind, os.path.join(processed_dir, "era5_wind_percentil.nc"), "anomalies_wind_combined.csv")

### 6b. Precipitation + drought -- per region, unoptimized (see note above)

In [ ]:
stage3_rain_times = {}
for region in REGIONS:
    print(f"\n=== {region['name']} (precipitation/drought) ===")
    t0 = time.perf_counter()
    anomalias_rain.procesar_anomalias_lluvia(
        shapefile_path=os.path.join(shapefiles_dir, region["shapefile"]),
        ruta=processed_dir,
        ruta_grib=raw_dir,
        ruta_salida=os.path.join(processed_dir, region["name"]),
    )
    stage3_rain_times[region["name"]] = time.perf_counter() - t0
    print(f"{region['name']}: {stage3_rain_times[region['name']]/60:.1f} min")

## 7. Summary

In [ ]:
import pandas as pd

rows = []
for region in REGIONS:
    region_dir = os.path.join(processed_dir, region["name"])
    for fname in ["anomalies_temperature_combined.csv", "anomalies_wind_combined.csv",
                  "anomalies_precipitation_combined.csv", "anomalies_drought_combined.csv"]:
        path = os.path.join(region_dir, fname)
        rows.append({
            "region": region["name"],
            "file": fname,
            "exists": os.path.exists(path),
            "rows": len(pd.read_csv(path)) if os.path.exists(path) else 0,
        })

summary = pd.DataFrame(rows)
print(f"Precipitation/drought stage total: {sum(stage3_rain_times.values())/60:.1f} min")
summary